# Reader note

This notebook is part of the `arXiv:2606.04091` reproduction workflow. It reproduces `Figure 2`, speed distributions for different halo models prescriptions and speed ranges.
Old execution outputs are intentionally cleared for release.


In [ ]:
import numpy as np
import script_helpers.script_VDF as vdf
import itertools
from scipy.integrate import simpson as simps
from scipy.optimize import root_scalar
import matplotlib.pyplot as plt

In [ ]:
### Conservative speed central values in standard prescription
v0_conservative = 220
v_esc_conservative = 544

# Ranges for uncertainty bands
v0_conservative_range = [200, 280]
v_esc_conservative_range = [450, 600]

v0_conservative_list = np.arange(v0_conservative_range[0], v0_conservative_range[1], 1.0)
v_esc_conservative_list = np.arange(v_esc_conservative_range[0], v_esc_conservative_range[1], 1.88)

### Aggressive speed central values in standard prescription
v0_aggressive = 238
v_esc_aggressive = 528

# Ranges for uncertainty bands
v0_aggressive_range = [236.5, 239.5]
v_esc_aggressive_range = [503, 552]

v0_aggressive_list = np.arange(v0_aggressive_range[0], v0_aggressive_range[1], 0.04)
v_esc_aggressive_list = np.arange(v_esc_aggressive_range[0], v_esc_aggressive_range[1], 0.66)

# Parameters
v_min = 0  # lower limit
v = np.linspace(v_min, 800, 1000) # integration parameters
p_val = 1.5  # p value for empirical distribution

In [ ]:
## Display the lengths of the lists and their contents (optional)
# print(len(v0_conservative_list), len(v_esc_conservative_list))
# print(f"v0_conservative_list: {v0_conservative_list}")
# print(f"v_esc_conservative_list: {v_esc_conservative_list}")

# print(len(v0_aggressive_list), len(v_esc_aggressive_list))
# print(f"v0_aggressive_list: {v0_aggressive_list}")
# print(f"v_esc_aggressive_list: {v_esc_aggressive_list}")

In [ ]:
### Function to compute the rms of an normalized distribution f(v)
def find_rms(f, v):
    """     Compute the standard deviation of the distribution.
    Args:
        f (array): Normalized distribution values.
        v (array): Speed values.
    Returns:
        float: Root mean square (rms) value of the distribution.
    """
    mean_sq = simps(v**2 * f, v)               # ⟨v^2⟩
    vrms = np.sqrt(mean_sq)                # rms = sqrt(⟨v^2⟩)
    return vrms

### Find rms speed for SHM
def objective_MB(v0, v, v_min, v_esc):
    dist = vdf.compute_distributions(vdf.f_MB, v_min, v0, v_esc)
    rms = find_rms(dist, v)
    return rms

### Find difference of rms speed for Tsallis and target rms(SHM)
def objective_tsallis(x, v, v_min, v_esc, trgt_rms):
    dist = vdf.compute_distributions(vdf.f_Tsallis, v_min, x, v_esc)
    rms = find_rms(dist, v)
    return rms - trgt_rms

### Find difference of rms speed for Empirical and target rms(SHM)
def objective_empirical(x, v, v_min, v_esc, trgt_rms, p):
    dist = vdf.compute_distributions(vdf.f_Empirical, v_min, x, v_esc, p=p_val)
    rms = find_rms(dist, v)
    return rms - trgt_rms

### Find rms speed for Empirical distribution in the infinite-v0 limit
def rms_empirical_infinite(v, v_min, v_esc, p):
    dist = vdf.compute_distributions(
        vdf.f_Empirical_infinite,  # function
        v_min,                 # v_min (positional!)
        np.inf,                # v0 → ∞
        v_esc,                 # escape speed
        p=p_val
    )
    return find_rms(dist, v)

### Find v0_tsa that minimizes rms-trgt_rms for a given v_esc
def match_v0_Tsallis(trgt_rms, v_esc):
    a, b = 100, 450  # bracket

    f_a = objective_tsallis(a, v, v_min, v_esc, trgt_rms)
    f_b = objective_tsallis(b, v, v_min, v_esc, trgt_rms)

    # If the function changes sign over the bracket, we can use root_scalar to find the exact match
    if f_a * f_b < 0:
        sol = root_scalar(
            objective_tsallis,
            args=(v, v_min, v_esc, trgt_rms),
            bracket=[a, b],
            method='brentq'
        )
        return sol.root, 0.0  # matched exactly
    else:
        # Scan bracket and find x minimizing |rms - trgt_rms|
        xs = np.linspace(a, b, 500)
        diffs = [(x, abs(objective_tsallis(x, v, v_min, v_esc, trgt_rms)), 
                   objective_tsallis(x, v, v_min, v_esc, trgt_rms)) for x in xs]
        best_x, min_abs_diff, signed_diff = min(diffs, key=lambda tup: tup[1])
        print(f"[INFO] Fallback match for v0={trgt_rms}, v_esc={v_esc} → v0_tsallis={best_x:.2f}, Δrms={signed_diff:.4f}")
        return best_x, signed_diff

### Find v0_emp that minimizes rms-trgt_rms for a given v_esc
def match_v0_Empirical(trgt_rms, v_esc):
    # infinite-v0 limit
    rms_inf = rms_empirical_infinite(v, v_min, v_esc, p_val)
    if trgt_rms > rms_inf:
        # no finite solution exists
        return np.inf, trgt_rms - rms_inf

    # attempt finite solution
    a, b = 10.0, 710.0   # bracket

    fa = objective_empirical(a, v, v_min, v_esc, trgt_rms, p_val)
    fb = objective_empirical(b, v, v_min, v_esc, trgt_rms, p_val)

    # If no sign change, treat as infinite-v0 case
    if fa * fb > 0:
        return np.inf, trgt_rms - rms_inf

    sol = root_scalar(
        objective_empirical,
        args=(v, v_min, v_esc, trgt_rms, p_val),
        bracket=[a, b],
        method="brentq"
    )

    return sol.root, 0.0

In [ ]:
### Conservative speed range
# Compute all three speed distributions in standard prescription
f1 = vdf.f_MB_norm(v, v0_conservative, v_esc_conservative) * 1e0
f2 = vdf.f_Tsallis_norm(v, v0_conservative, v_esc_conservative) * 1e0  
f3 = vdf.f_Empirical_norm(v, v0_conservative, v_esc_conservative, p_val) * 1e0

# Compute the rms matched tsallis and empirical speed distributions
# 1. Target rms value for matching is set equal to the rms of the MB distribution at v0_conservative and v_esc_conservative
trgt_rms = objective_MB(v0_conservative, v, v_min, v_esc_conservative)
# 2.Find matched v0 values
v0_Tsallis_matched, diff_tsallis = match_v0_Tsallis(trgt_rms, v_esc_conservative)
v0_Empirical_matched, diff_empirical = match_v0_Empirical(trgt_rms, v_esc_conservative)
# 3. Compute the distributions
f_Tsallis = vdf.compute_distributions(vdf.f_Tsallis, v_min, v0_Tsallis_matched, v_esc_conservative)
f_Empirical = vdf.compute_distributions(vdf.f_Empirical, v_min, v0_Empirical_matched, v_esc_conservative, p=p_val)

In [ ]:
### Conservative speed range
## Display results (optional)
# print('Matched Velocities such that the distributions have the same v_rms in conservative speed range')

# print(
#     f"{'v0':>8} "
#     f"{'v_esc':>8} "
#     f"{'v_rms':>8} "
#     f"{'v0_MB_matched':>16} "
#     f"{'v0_Tsallis_matched':>20} "
#     f"{'delta_rms_TSA':>20} "
#     f"{'v0_Empirical_matched':>24} "
#     f"{'delta_rms_EMP':>20}"
# )

# print("-" * 128)

# print(
#     f"{v0_conservative:8.2f} "
#     f"{v_esc_conservative:8.2f} "
#     f"{trgt_rms:8.2f} "
#     f"{v0_conservative:16.2f} "
#     f"{v0_Tsallis_matched:20.2f} "
#     f"{diff_tsallis:20.2f} "
#     f"{v0_Empirical_matched:24.2f} "
#     f"{diff_empirical:20.2f}"
# )

In [ ]:
# color options
model_colors = {"SHM": "#3A86FF", "TSA": "#FF006E", "EMP": "#8338EC"}

## Making the matplotlib plots look nicer
settings = {
    # 'figure.constrained_layout.use': True,
    # 'mathtext.fontset': 'stix',
    # 'font.family': 'STIXGeneral',
    # LaTeX-like fonts
    'mathtext.fontset': 'cm',
    # 'font.family': 'serif',
    # 'font.serif': ['Computer Modern Roman'],
    # 'mathtext.fontset': 'dejavuserif',
    'font.family': 'DejaVu Serif',
    'font.size':15,
    # 'axes.labelsize': 'large',
    'lines.markersize': 5,
    'axes.linewidth':2.0,
    'xtick.major.size':8.0,
    'xtick.minor.size':4.0,
    'xtick.major.width':1.5,
    'xtick.minor.width':1.0,
    'xtick.direction':'in', 
    'xtick.minor.visible':True,
    'xtick.top':True,
    'ytick.major.size':8.0,
    'ytick.minor.size':4.0,
    'ytick.major.width':1.5,
    'ytick.minor.width':1.0,
    'ytick.direction':'in', 
    'ytick.minor.visible':True,
    'ytick.right':True,
    'contour.linewidth':3.0,
    'savefig.bbox': 'tight',
    'savefig.dpi': 200,
}

plt.rcParams.update(**settings) 

In [ ]:
### Conservative speed range
### Plotting the speed distribution function curves in standard and rms matching prescription
fig, ax = plt.subplots(figsize=(8, 7))

# Plot nominal lines
ax.plot(v, f1, color=model_colors["SHM"], lw=2.5, linestyle='-', label=r'SHM')
ax.plot(v, f2, color=model_colors["TSA"], lw=1.5, linestyle='--')
ax.plot(v, f3, color=model_colors["EMP"], lw=1.5, linestyle='--')


ax.plot(v, f_Tsallis, color=model_colors["TSA"], lw = 2.5, linestyle='-', label = r'Tsallis')
ax.plot(v, f_Empirical, color=model_colors["EMP"], lw = 2.5, linestyle='-', label=rf'Empirical')

# Vertical lines
ax.axvline(v0_conservative, color='gray', linestyle='--')
ax.axvline(v_esc_conservative, color='gray', linestyle=':')
ax.text(v0_conservative + 7, ax.get_ylim()[1]*0.3, fr'$v_c = {v0_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
ax.text(v_esc_conservative + 7, ax.get_ylim()[1]*0.3, fr'$v_{{\mathrm{{esc}}}} = {v_esc_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)

ax.set_xlabel('$v$ ($\mathrm{km} \, \mathrm{s}^{-1}$)', fontsize=22)
ax.set_ylabel(r'$4\pi v^2f_{gal}(v)$ ($\mathrm{km}^{-1} \, \mathrm{s}$)', fontsize=22)
ax.set_xticks(np.arange(0, 801, 100))
ax.set_xlim(0, 602)
ax.set_ylim(0, 0.006)
ax.grid(False)
ax.legend(loc='upper right', frameon=True, fontsize=18)
plt.tight_layout()
plt.show()

In [ ]:
### Conservative speed range
# Evaluate bands for conservative range in standard prescription
v0_conservative_vals = np.linspace(v0_conservative_range[0], v0_conservative_range[1], 80)
vesc_conservative_vals = np.linspace(v_esc_conservative_range[0], v_esc_conservative_range[1], 80)

f1_conservative_min, f1_conservative_max = vdf.get_band(vdf.f_MB_norm, v, v0_conservative_vals, vesc_conservative_vals)
f2_conservative_min, f2_conservative_max = vdf.get_band(vdf.f_Tsallis_norm, v, v0_conservative_vals, vesc_conservative_vals)
f3_conservative_min, f3_conservative_max = vdf.get_band(vdf.f_Empirical_norm, v, v0_conservative_vals, vesc_conservative_vals, p=p_val)

# Evaluate bands for conservative range in rms matching prescription
# Prepare storage for matched velocities
matched_results_conservative = []

# Loop over all combinations
for v_esc0,v00 in itertools.product(v_esc_conservative_list, v0_conservative_list):
    trgt_rms = objective_MB(v00, v, v_min, v_esc0)  # Target rms value for matching is set to be equal to the rms of the MB distribution at v00 and v_esc0
    v0_MB_matched = v00  
    v0_Tsallis_matched, diff_tsallis = match_v0_Tsallis(trgt_rms, v_esc0)
    v0_Empirical_matched, diff_empirical = match_v0_Empirical(trgt_rms, v_esc0)

    # Append to results
    matched_results_conservative.append({
        'v0': v00,
        'v_esc': v_esc0,
        'v_rms': trgt_rms,
        'v0_MB_matched': v0_MB_matched,
        'v0_Tsallis_matched': v0_Tsallis_matched,
        'delta_rms_TSA': diff_tsallis,
        'v0_Empirical_matched': v0_Empirical_matched,
        'delta_rms_EMP': diff_empirical
    })

In [ ]:
## Display results (optional)
# print('Matched Velocities for Different v0 and v_esc Combinations such that v_rms  = v0 in conservative speed range')
# print(f"{'v0':>8} {'v_esc':>8} {'v_rms':>8}{'v0_MB_matched':>16} {'v0_Tsallis_matched':>20} {'delta_rms_TSA':>20} {'v0_Empirical_matched':>24} {'delta_rms_EMP':>20}")
# print("-" * 128)

# for result in matched_results_conservative:
#     print(f"{result['v0']:8.2f} {result['v_esc']:8.2f} {result['v_rms']:8.2f} {result['v0_MB_matched']:16.2f} "
#           f"{result['v0_Tsallis_matched']:20.2f} {result['delta_rms_TSA']:20.2f} "
#           f"{result['v0_Empirical_matched']:24.2f} {result['delta_rms_EMP']:20.2f}")

In [ ]:
### Conservative speed range
# Preallocate arrays to collect all distributions
all_MB = []
all_Tsallis = []
all_Empirical = []

# Compute distributions for all combinations
for result in matched_results_conservative:
    v0_shm = result['v0_MB_matched']
    v0_tsallis = result['v0_Tsallis_matched']
    v0_empirical = result['v0_Empirical_matched']
    v_esc0 = result['v_esc']
    
    f_MB = vdf.compute_distributions(vdf.f_MB, v_min, v0_shm, v_esc0)
    f_Tsallis = vdf.compute_distributions(vdf.f_Tsallis, v_min, v0_tsallis, v_esc0)
    f_Emp = vdf.compute_distributions(vdf.f_Empirical, v_min, v0_empirical, v_esc0, p=p_val)

    all_MB.append(f_MB)
    all_Tsallis.append(f_Tsallis)
    all_Empirical.append(f_Emp)

# Convert to arrays for easier processing
all_MB = np.array(all_MB)
all_Tsallis = np.array(all_Tsallis)
all_Empirical = np.array(all_Empirical)

# Calculate min and max envelopes
MB_min, MB_max = np.min(all_MB, axis=0), np.max(all_MB, axis=0)
Tsallis_min, Tsallis_max = np.min(all_Tsallis, axis=0), np.max(all_Tsallis, axis=0)
Emp_min, Emp_max = np.min(all_Empirical, axis=0), np.max(all_Empirical, axis=0)

In [ ]:
### Conservative speed range
# Plotting the speed distribution function envelopes in standard and rms matching prescription
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 13), sharex=True, constrained_layout=False)
fig.subplots_adjust(hspace=-0.0)

# Upper panel
ax1.fill_between(v, f1_conservative_min, f1_conservative_max, color=model_colors["SHM"], alpha=0.4)
ax1.fill_between(v, f2_conservative_min, f2_conservative_max, color=model_colors["TSA"], alpha=0.4)
ax1.fill_between(v, f3_conservative_min, f3_conservative_max, color=model_colors["EMP"], alpha=0.4)

# Vertical lines
ax1.axvline(v0_conservative, color='gray', linestyle='--')
ax1.axvline(v_esc_conservative, color='gray', linestyle=':')
ax1.text(v0_conservative + 7, ax1.get_ylim()[1]*0.3, fr'$v_c = {v0_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
ax1.text(v_esc_conservative + 7, ax1.get_ylim()[1]*0.3, fr'$v_{{\mathrm{{esc}}}} = {v_esc_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
# Title
ax1.text(v0_conservative - 195, ax1.get_ylim()[1]*1.0, 'standard prescription: Conservative range', rotation=0, color='black', va='center', fontsize=19,
        bbox=dict(
        facecolor='white',
        edgecolor='none',   
        alpha=1.0,          
        boxstyle='round,pad=0.3'  
    ))

ax1.set_xticks(np.arange(0, 801, 100))
ax1.set_xlim(0, 602)
ax1.set_ylim(0, 0.006)

# Lower panel
ax2.fill_between(v, MB_min, MB_max, color=model_colors["SHM"], alpha=0.4)
ax2.fill_between(v, Tsallis_min, Tsallis_max, color=model_colors["TSA"], alpha=0.4)
ax2.fill_between(v, Emp_min, Emp_max, color=model_colors["EMP"], alpha=0.4)

# Vertical lines
ax2.axvline(v0_conservative, color='gray', linestyle='--')
ax2.axvline(v_esc_conservative, color='gray', linestyle=':')
ax2.text(v0_conservative + 7, ax2.get_ylim()[1]*0.3, fr'$v_c = {v0_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
ax2.text(v_esc_conservative + 7, ax2.get_ylim()[1]*0.3, fr'$v_{{\mathrm{{esc}}}} = {v_esc_conservative} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
# Title
ax2.text(v0_conservative - 205, ax1.get_ylim()[1]*0.92, 'rms-matching prescription: Conservative range', rotation=0, color='black', va='center', fontsize=17.6,
        bbox=dict(
        facecolor='white',
        edgecolor='none',   
        alpha=1.0,          
        boxstyle='round,pad=0.3'  
    ))

ax2.set_xlabel('$v$ ($\mathrm{km} \, \mathrm{s}^{-1}$)', fontsize=20)
ax2.set_xticks(np.arange(0, 801, 100))
ax2.set_xlim(0, 602)
ax2.set_ylim(0, 0.006)
ax2.set_yticks([0.000, 0.001, 0.002, 0.003, 0.004, 0.005])
plt.show()


In [ ]:
### Aggressive speed range
# Evaluate bands for aggressive range in standard prescription
v0_aggressive_vals = np.linspace(v0_aggressive_range[0], v0_aggressive_range[1], 80) #, 10)
vesc_aggressive_vals = np.linspace(v_esc_aggressive_range[0], v_esc_aggressive_range[1], 80) #, 10)

f1_aggressive_min, f1_aggressive_max = vdf.get_band(vdf.f_MB_norm, v, v0_aggressive_vals, vesc_aggressive_vals)
f2_aggressive_min, f2_aggressive_max = vdf.get_band(vdf.f_Tsallis_norm, v, v0_aggressive_vals, vesc_aggressive_vals)
f3_aggressive_min, f3_aggressive_max = vdf.get_band(vdf.f_Empirical_norm, v, v0_aggressive_vals, vesc_aggressive_vals, p=p_val)

# Evaluate bands for aggressive range in rms matching prescription
# Prepare storage for matched velocities
matched_results_aggressive = []

# Loop over all combinations
for v_esc0,v00 in itertools.product(v_esc_aggressive_list, v0_aggressive_list):
    trgt_rms = objective_MB(v00, v, v_min, v_esc0)  # Target rms value for matching is set to be equal to the rms of the MB distribution at v00 and v_esc0
    v0_MB_matched = v00  
    v0_Tsallis_matched, diff_tsallis = match_v0_Tsallis(trgt_rms, v_esc0)
    v0_Empirical_matched, diff_empirical = match_v0_Empirical(trgt_rms, v_esc0)

    # Append to results
    matched_results_aggressive.append({
        'v0': v00,
        'v_esc': v_esc0,
        'v_rms': trgt_rms,
        'v0_MB_matched': v0_MB_matched,
        'v0_Tsallis_matched': v0_Tsallis_matched,
        'delta_rms_TSA': diff_tsallis,
        'v0_Empirical_matched': v0_Empirical_matched,
        'delta_rms_EMP': diff_empirical
    })

In [ ]:
## Display results (optional)
# print('Matched Velocities for Different v0 and v_esc Combinations such that v_rms  = v0 in aggressive speed range')
# print(f"{'v0':>8} {'v_esc':>8} {'v_rms':>8}{'v0_MB_matched':>16} {'v0_Tsallis_matched':>20} {'delta_rms_TSA':>20} {'v0_Empirical_matched':>24} {'delta_rms_EMP':>20}")
# print("-" * 128)

# for result in matched_results_aggressive:
#     print(f"{result['v0']:8.2f} {result['v_esc']:8.2f} {result['v_rms']:8.2f} {result['v0_MB_matched']:16.2f} "
#           f"{result['v0_Tsallis_matched']:20.2f} {result['delta_rms_TSA']:20.2f} "
#           f"{result['v0_Empirical_matched']:24.2f} {result['delta_rms_EMP']:20.2f}")

In [ ]:
### Aggressive speed range
# Preallocate arrays to collect all distributions
all_MB = []
all_Tsallis = []
all_Empirical = []

# Compute distributions for all combinations
for result in matched_results_aggressive:
    v0_shm = result['v0_MB_matched']
    v0_tsallis = result['v0_Tsallis_matched']
    v0_empirical = result['v0_Empirical_matched']
    v_esc0 = result['v_esc']
    
    f_MB = vdf.compute_distributions(vdf.f_MB, v_min, v0_shm, v_esc0)
    f_Tsallis = vdf.compute_distributions(vdf.f_Tsallis, v_min, v0_tsallis, v_esc0)
    f_Emp = vdf.compute_distributions(vdf.f_Empirical, v_min, v0_empirical, v_esc0, p=p_val)

    all_MB.append(f_MB)
    all_Tsallis.append(f_Tsallis)
    all_Empirical.append(f_Emp)

# Convert to arrays for easier processing
all_MB = np.array(all_MB)
all_Tsallis = np.array(all_Tsallis)
all_Empirical = np.array(all_Empirical)

# Calculate min and max envelopes
MB_min, MB_max = np.min(all_MB, axis=0), np.max(all_MB, axis=0)
Tsallis_min, Tsallis_max = np.min(all_Tsallis, axis=0), np.max(all_Tsallis, axis=0)
Emp_min, Emp_max = np.min(all_Empirical, axis=0), np.max(all_Empirical, axis=0)

In [ ]:
### Aggressive speed range
# Plotting the speed distribution function envelopes in standard and rms matching prescription
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 13), sharex=True, constrained_layout=False)
fig.subplots_adjust(hspace=-0.0)

# Upper panel
ax1.fill_between(v, f1_aggressive_min, f1_aggressive_max, color=model_colors["SHM"], alpha=0.4)
ax1.fill_between(v, f2_aggressive_min, f2_aggressive_max, color=model_colors["TSA"], alpha=0.4)
ax1.fill_between(v, f3_aggressive_min, f3_aggressive_max, color=model_colors["EMP"], alpha=0.4)

# Vertical lines
ax1.axvline(v0_aggressive, color='gray', linestyle='--')
ax1.axvline(v_esc_aggressive, color='gray', linestyle=':')
ax1.text(v0_aggressive + 7, ax1.get_ylim()[1]*0.3, fr'$v_c = {v0_aggressive} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
ax1.text(v_esc_aggressive + 7, ax1.get_ylim()[1]*0.3, fr'$v_{{\mathrm{{esc}}}} = {v_esc_aggressive} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
# Title
ax1.text(v0_aggressive - 195, ax1.get_ylim()[1]*1.15, 'standard prescription: Aggressive range', rotation=0, color='black', va='center', fontsize=19,
        bbox=dict(
        facecolor='white',
        edgecolor='none',   
        alpha=1.0,          
        boxstyle='round,pad=0.3'  
    ))

ax1.set_xticks(np.arange(0, 801, 100))
ax1.set_xlim(0, 602)
ax1.set_ylim(0, 0.006)

# Lower panel
ax2.fill_between(v, MB_min, MB_max, color=model_colors["SHM"], alpha=0.4)
ax2.fill_between(v, Tsallis_min, Tsallis_max, color=model_colors["TSA"], alpha=0.4)
ax2.fill_between(v, Emp_min, Emp_max, color=model_colors["EMP"], alpha=0.4)

# Vertical lines
ax2.axvline(v0_aggressive, color='gray', linestyle='--')
ax2.axvline(v_esc_aggressive, color='gray', linestyle=':')
ax2.text(v0_aggressive + 7, ax2.get_ylim()[1]*0.3, fr'$v_c = {v0_aggressive} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
ax2.text(v_esc_aggressive + 7, ax2.get_ylim()[1]*0.3, fr'$v_{{\mathrm{{esc}}}} = {v_esc_aggressive} \, \mathrm{{km}} \, \mathrm{{s}}^{{-1}}$', rotation=90, color='gray', va='center', fontsize=18)
# Title
ax2.text(v0_aggressive - 205, ax1.get_ylim()[1]*0.92, 'rms-matching prescription: Aggressive range', rotation=0, color='black', va='center', fontsize=17.6,
        bbox=dict(
        facecolor='white',
        edgecolor='none',   
        alpha=1.0,          
        boxstyle='round,pad=0.3'  
    ))

ax2.set_xlabel('$v$ ($\mathrm{km} \, \mathrm{s}^{-1}$)', fontsize=20)
ax2.set_xticks(np.arange(0, 801, 100))
ax2.set_xlim(0, 602)
ax2.set_ylim(0, 0.006)
ax2.set_yticks([0.000, 0.001, 0.002, 0.003, 0.004, 0.005])
plt.show()